# Simple Agent Use

A minimal, self-contained example of an **agentic loop**: call an LLM, run any
tool calls it makes, feed the results back in, and repeat until it's done.

The model plans and tracks its own progress via two tools:

- `create_checklist(descriptions)` — plan the steps
- `mark_complete(index, completion_notes)` — check them off as it goes

Progress is rendered live in the terminal with [`rich`](https://github.com/Textualize/rich).

**call model → run tools → feed results back → repeat until done**

*Adapted from Ed Donner's [Agents course](https://github.com/ed-donner/agents).*

## Setup

1. `pip install -r requirements.txt`
2. Copy `.env.example` to `.env` and add your OpenRouter API key
   (get one at https://openrouter.ai/)


In [49]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
load_dotenv(override=True)

True

In [50]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [51]:
openai = client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPEN_ROUTER_API_KEY")
)

## Checklist tools

Plain Python functions, plus the JSON schemas the model needs to call them.

In [52]:
# Some lists!

checklist = []
completed = []

In [53]:
def get_checklist_report() -> str:
    result = ""
    for index, item in enumerate(checklist):
        if completed[index]:
            result += f"Checklist #{index + 1}: [green][strike]{item}[/strike][/green]\n"
        else:
            result += f"Checklist #{index + 1}: {item}\n"
    show(result)
    return result

In [54]:
def create_checklist(descriptions: list[str]) -> str:
    checklist.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_checklist_report()

In [55]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(checklist):
        completed[index - 1] = True
    else:
        return "No checklist at this index."
    Console().print(completion_notes)
    return get_checklist_report()

A quick manual sanity check that the checklist functions behave as expected, before wiring them up as tools.

In [56]:
checklist, completed = [], []

create_checklist(["Buy groceries", "Finish week 1", "Eat banana"])

Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: Buy groceries\nChecklist #2: Finish week 1\nChecklist #3: Eat banana\n'

In [57]:
mark_complete(1, "bought")

bought

Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: [green][strike]Buy groceries[/strike][/green]\nChecklist #2: Finish week 1\nChecklist #3: Eat banana\n'

## Tool schemas

These describe the functions above to the model, in OpenAI's function-calling format.

In [58]:
create_checklist_json = {
    "name": "create_checklist",
    "description": "Add new checklist from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions of checklist items'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [59]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the checklist item at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the checklist item to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the checklist item in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [60]:
tools = [{"type": "function", "function": create_checklist_json},
        {"type": "function", "function": mark_complete_json}]

## The agent loop

`handle_tool_calls` executes whatever the model asks for; `loop` is the actual agentic while-loop.

In [61]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [62]:
def loop(messages):
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=tools)
    show(response.choices[0].message.content)

## Try it

Give the agent a problem that needs multiple steps and an estimate.

In [63]:
system_message = """
You are given a problem to solve, by using your checklist tools to plan a list of steps, then carrying out each step in turn.
Now create a plan, set the checklist, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [64]:
checklist, completed = [], []
loop(messages)

Checklist #1: Identify the starting points and speeds of the trains.
Checklist #2: Determine the time difference between the departures of the two trains.
Checklist #3: Calculate the distance traveled by the Boston train until the New York train departs.
Checklist #4: Set up the equation to find the time when the two trains meet.
Checklist #5: Solve the equation for the meeting time.

The train from Boston travels at 60 mph, and the train from New York travels at 80 mph. They are moving toward each
other.

Checklist #1: Identify the starting points and speeds of the trains.
Checklist #2: Determine the time difference between the departures of the two trains.
Checklist #3: Calculate the distance traveled by the Boston train until the New York train departs.
Checklist #4: Set up the equation to find the time when the two trains meet.
Checklist #5: Solve the equation for the meeting time.

The Boston train leaves at 2:00 pm and the New York train leaves at 3:00 pm, giving a 1-hour difference.

Checklist #1: Identify the starting points and speeds of the trains.
Checklist #2: Determine the time difference between the departures of the two trains.
Checklist #3: Calculate the distance traveled by the Boston train until the New York train departs.
Checklist #4: Set up the equation to find the time when the two trains meet.
Checklist #5: Solve the equation for the meeting time.

...

Let t be the time in hours after 3:00 pm when they meet. Distance covered by Boston train is 60(t + 1) and by New 
York train is 80t. Setting them equal: 60(t + 1) + 80t = 60 + 80t.

Checklist #1: Identify the starting points and speeds of the trains.
Checklist #2: Determine the time difference between the departures of the two trains.
Checklist #3: Calculate the distance traveled by the Boston train until the New York train departs.
Checklist #4: Set up the equation to find the time when the two trains meet.
Checklist #5: Solve the equation for the meeting time.

Solving the equation: 60(t + 1) = 80t --> 60t + 60 = 80t --> 60 = 20t --> t = 3. Thus, they meet at 3:00 pm + 3 
hours = 6:00 pm.

Checklist #1: Identify the starting points and speeds of the trains.
Checklist #2: Determine the time difference between the departures of the two trains.
Checklist #3: Calculate the distance traveled by the Boston train until the New York train departs.
Checklist #4: Set up the equation to find the time when the two trains meet.
Checklist #5: Solve the equation for the meeting time.

The two trains meet at 6:00 pm.